# Stage 4: OVB + Sensemakr

**Goal**: Decompose the halo effect mathematically using Omitted Variable Bias (OVB), verify the attractiveness partial coefficient via FWL, and quantify robustness to omitted confounders using Sensemakr.

**Analysis order**:
1. **Pairwise OVB** — raw halo mechanism (each non-attr trait vs attr)
2. **Controlled OVB** — robustness after controlling all other perceived traits
3. **FWL** — bridge to attr partial effect in the full model
4. **Sensemakr** — sensitivity of attr effect to unobserved confounding


In [25]:
# ── Working directory → project root ─────────────────────────────────────────
import os
from pathlib import Path

def _project_root(marker="README.md"):
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / marker).exists():
            return p
    raise FileNotFoundError("Project root not found")

os.chdir(_project_root())
print("Working directory:", Path.cwd())

# ── Imports ───────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import statsmodels.formula.api as smf
import statsmodels.api as sm

# Create output directories
Path("tables").mkdir(exist_ok=True)
Path("figures").mkdir(exist_ok=True)

TRAITS      = ["sinc", "intel", "fun", "amb", "shar"]
TRAIT_LABEL = {"sinc": "Sincerity", "intel": "Intelligence",
               "fun": "Fun", "amb": "Ambition", "shar": "Shared Interests"}
CI_MULT     = 1.96  # 95% CI


Working directory: /Users/chen/Study/UCB/STAT230a/speed-dating-halo


In [26]:
df = pd.read_parquet("data/clean/cleaned.parquet")
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

# Verify required columns
required = ["dec", "attr"] + TRAITS + ["iid", "pid", "gender", "samerace", "age_diff"]
missing  = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")
print(f"All required columns present.")

# Show missingness for analysis columns
print("\nNull counts:")
print(df[["dec","attr"] + TRAITS + ["iid","gender","samerace","age_diff"]].isnull().sum().to_string())


Loaded: 5,742 rows × 114 columns
All required columns present.

Null counts:
dec           0
attr          0
sinc          0
intel         0
fun           0
amb           0
shar          0
iid           0
gender        0
samerace      0
age_diff    142


---
## Part A: Pairwise OVB Decomposition

For each non-attr trait, fit three regressions **on the same sample** (per-trait dropna):

- **Short**: `dec ~ trait`
- **Long**: `dec ~ trait + attr`
- **Auxiliary**: `attr ~ trait`  ← *this is the OVB auxiliary regression, not another short regression*

OVB identity (exact under OLS/LPM):

$$\hat{\alpha}_1 = \hat{\beta}_1 + \hat{\beta}_2 \cdot \hat{\gamma}$$

Stage 4 reports **coefficient change / shrinkage magnitude**, not a significance test on the shrinkage.


In [27]:
results_a = []

for trait in TRAITS:
    # Same sample for all three regressions
    df_trait = df[["dec", "attr", trait, "iid"]].dropna().copy()
    n = len(df_trait)

    # Short regression: dec ~ trait
    short = smf.ols(f"dec ~ {trait}", data=df_trait).fit(
        cov_type="cluster", cov_kwds={"groups": df_trait["iid"].values})
    # Long regression: dec ~ trait + attr
    long_ = smf.ols(f"dec ~ {trait} + attr", data=df_trait).fit(
        cov_type="cluster", cov_kwds={"groups": df_trait["iid"].values})
    # Auxiliary regression: attr ~ trait (plain OLS; no cluster needed)
    aux   = smf.ols(f"attr ~ {trait}", data=df_trait).fit()

    sc  = short.params[trait]
    lc  = long_.params[trait]
    ac  = long_.params["attr"]
    gm  = aux.params[trait]
    ovb = ac * gm
    err = abs(sc - (lc + ovb))

    results_a.append({
        "trait":             trait,
        "short_coef":        sc,
        "long_coef":         lc,
        "attr_coef_in_long": ac,
        "gamma_from_aux":    gm,
        "ovb_component":     ovb,
        "formula_error":     err,
        "absolute_change":   abs(lc - sc),
        "relative_change_pct": abs(lc - sc) / abs(sc) * 100 if sc != 0 else None,
        "direction_change":  bool(np.sign(sc) != np.sign(lc)),
        "n":                 n,
    })

table04a = pd.DataFrame(results_a)
table04a.to_csv("tables/table04a_pairwise_ovb.csv", index=False)

# ── Display ────────────────────────────────────────────────────────────────────
print("=== Pairwise OVB Decomposition ===")
display_cols = ["trait","short_coef","long_coef","attr_coef_in_long",
                "gamma_from_aux","ovb_component","formula_error",
                "absolute_change","relative_change_pct","direction_change"]
print(table04a[display_cols].to_string(index=False, float_format=lambda x: f"{x:.5f}"))

print("\n=== OVB Formula Verification ===")
all_pass = True
for _, row in table04a.iterrows():
    status = "PASS" if row["formula_error"] < 0.001 else "FAIL"
    if status == "FAIL":
        all_pass = False
    print(f"  {row['trait']:5s}: formula_error = {row['formula_error']:.2e}  [{status}]")
print(f"\nAll formula checks passed: {all_pass}")


=== Pairwise OVB Decomposition ===
trait  short_coef  long_coef  attr_coef_in_long  gamma_from_aux  ovb_component  formula_error  absolute_change  relative_change_pct  direction_change
 sinc     0.05839    0.00376            0.12057         0.45307        0.05463        0.00000          0.05463             93.55904             False
intel     0.06873    0.01150            0.11836         0.48352        0.05723        0.00000          0.05723             83.27397             False
  fun     0.10310    0.04987            0.09216         0.57761        0.05323        0.00000          0.05323             51.62872             False
  amb     0.04960    0.00339            0.12087         0.38231        0.04621        0.00000          0.04621             93.16656             False
 shar     0.09287    0.05168            0.09349         0.44061        0.04119        0.00000          0.04119             44.35366             False

=== OVB Formula Verification ===
  sinc : formula_error = 2.78e-

In [28]:
fig, ax = plt.subplots(figsize=(9, 5))

x      = np.arange(len(TRAITS))
width  = 0.35
labels = [TRAIT_LABEL[t] for t in TRAITS]

bars_short = ax.bar(x - width/2,
                    table04a["short_coef"].values,
                    width, label="Short (dec ~ trait)",
                    color="#1f77b4", alpha=0.85)
bars_long  = ax.bar(x + width/2,
                    table04a["long_coef"].values,
                    width, label="Long (dec ~ trait + attr)",
                    color="#ff7f0e", alpha=0.85)

# Annotate shrinkage %
for i, row in table04a.iterrows():
    if row["relative_change_pct"] is not None:
        ypos = max(abs(row["short_coef"]), abs(row["long_coef"])) + 0.004
        ax.text(i, ypos,
                f"−{row['relative_change_pct']:.0f}%",
                ha="center", va="bottom", fontsize=9, color="#333333")

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylabel("OLS Coefficient (LPM)", fontsize=11)
ax.set_title("Pairwise OVB: Coefficient Shrinkage After Adding Attractiveness",
             fontsize=12, fontweight="bold")
ax.axhline(0, color="black", linewidth=0.8)
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))
fig.tight_layout()
fig.savefig("figures/fig07_pairwise_ovb_shrinkage.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: figures/fig07_pairwise_ovb_shrinkage.png")


Saved: figures/fig07_pairwise_ovb_shrinkage.png


---
## Part B: Controlled / Multivariate OVB

Fit all five non-attr traits together and check how much adding `attr` changes each partial coefficient.

```
short_controlled: dec ~ sinc + intel + fun + amb + shar
long_controlled:  dec ~ sinc + intel + fun + amb + shar + attr
aux_controlled:   attr ~ sinc + intel + fun + amb + shar
```

Matrix OVB identity:

$$\alpha_X = \beta_X + \beta_{\text{attr}} \cdot \Pi$$

where:
- $\alpha_X$: coefficient vector in `short_controlled` (5 traits)
- $\beta_X$: coefficient vector in `long_controlled` (5 traits)
- $\beta_{\text{attr}}$: scalar coefficient on `attr` in `long_controlled`
- $\Pi$: coefficient vector from `aux_controlled` (`attr ~ all traits`)
- Implied OVB component for trait $j$: $\beta_{\text{attr}} \times \Pi_j$

This is a **matrix-version** OVB — do not apply the scalar formula directly.


In [29]:
df_ctrl = df[["dec", "attr"] + TRAITS + ["iid"]].dropna().copy()
ctrl_formula = " + ".join(TRAITS)

short_ctrl = smf.ols(f"dec ~ {ctrl_formula}", data=df_ctrl).fit(
    cov_type="cluster", cov_kwds={"groups": df_ctrl["iid"].values})
long_ctrl  = smf.ols(f"dec ~ {ctrl_formula} + attr", data=df_ctrl).fit(
    cov_type="cluster", cov_kwds={"groups": df_ctrl["iid"].values})
aux_ctrl   = smf.ols(f"attr ~ {ctrl_formula}", data=df_ctrl).fit()

attr_coef_ctrl = long_ctrl.params["attr"]
print(f"attr coef in long_controlled = {attr_coef_ctrl:.5f}")

results_b = []
for trait in TRAITS:
    sp  = short_ctrl.params[trait]
    lp  = long_ctrl.params[trait]
    ag  = aux_ctrl.params[trait]
    imp = attr_coef_ctrl * ag
    fe  = abs(sp - (lp + imp))
    results_b.append({
        "trait":               trait,
        "short_partial_coef":  sp,
        "long_partial_coef":   lp,
        "change":              lp - sp,
        "absolute_change":     abs(lp - sp),
        "relative_change_pct": abs(lp - sp) / abs(sp) * 100 if sp != 0 else None,
        "direction_change":    bool(np.sign(sp) != np.sign(lp)),
        "aux_gamma":           ag,
        "implied_ovb_component": imp,
        "formula_error":       fe,
    })

table04b = pd.DataFrame(results_b)
table04b.to_csv("tables/table04b_controlled_ovb.csv", index=False)

print("\n=== Controlled OVB: Partial Coefficient Changes ===")
display_b = ["trait","short_partial_coef","long_partial_coef","change",
             "absolute_change","relative_change_pct","direction_change",
             "aux_gamma","implied_ovb_component","formula_error"]
print(table04b[display_b].to_string(index=False, float_format=lambda x: f"{x:.5f}"))

print("\n=== Matrix OVB Formula Verification ===")
all_pass_b = True
for _, row in table04b.iterrows():
    status = "PASS" if row["formula_error"] < 0.001 else "FAIL"
    if status == "FAIL":
        all_pass_b = False
    print(f"  {row['trait']:5s}: formula_error = {row['formula_error']:.2e}  [{status}]")
print(f"\nAll controlled OVB checks passed: {all_pass_b}")


attr coef in long_controlled = 0.08648

=== Controlled OVB: Partial Coefficient Changes ===
trait  short_partial_coef  long_partial_coef   change  absolute_change  relative_change_pct  direction_change  aux_gamma  implied_ovb_component  formula_error
 sinc            -0.00915           -0.01911 -0.00996          0.00996            108.80968             False    0.11516                0.00996        0.00000
intel             0.01264            0.00833 -0.00431          0.00431             34.07830             False    0.04980                0.00431        0.00000
  fun             0.07139            0.03805 -0.03334          0.03334             46.70675             False    0.38558                0.03334        0.00000
  amb            -0.02249           -0.02348 -0.00099          0.00099              4.40877             False    0.01146                0.00099        0.00000
 shar             0.05986            0.04546 -0.01440          0.01440             24.05064             False    

In [30]:
fig, ax = plt.subplots(figsize=(9, 5))

x      = np.arange(len(TRAITS))
width  = 0.35
labels = [TRAIT_LABEL[t] for t in TRAITS]

bars_s = ax.bar(x - width/2,
                table04b["short_partial_coef"].values,
                width, label="Short controlled (without attr)",
                color="#2ca02c", alpha=0.85)
bars_l = ax.bar(x + width/2,
                table04b["long_partial_coef"].values,
                width, label="Long controlled (with attr)",
                color="#d62728", alpha=0.85)

for i, row in table04b.iterrows():
    if row["relative_change_pct"] is not None:
        ypos = max(abs(row["short_partial_coef"]),
                   abs(row["long_partial_coef"])) + 0.003
        ax.text(i, ypos,
                f"−{row['relative_change_pct']:.0f}%",
                ha="center", va="bottom", fontsize=9, color="#333333")

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylabel("OLS Partial Coefficient (LPM)", fontsize=11)
ax.set_title("Controlled OVB: Partial Coefficient Shrinkage After Adding Attractiveness",
             fontsize=12, fontweight="bold")
ax.axhline(0, color="black", linewidth=0.8)
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))
fig.tight_layout()
fig.savefig("figures/fig08_controlled_ovb_shrinkage.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: figures/fig08_controlled_ovb_shrinkage.png")


Saved: figures/fig08_controlled_ovb_shrinkage.png


---
## Part C: FWL Verification

**FWL vs OVB**:
- OVB: how does omitting `attr` distort the coefficients on sinc/intel/fun/amb/shar?
- FWL: how is the `attr` partial coefficient in the full model identified?

FWL (Frisch–Waugh–Lovell) says: the coefficient on `attr` in
`dec ~ attr + sinc + intel + fun + amb + shar`
equals the coefficient in `resid_y ~ resid_attr`, where both residuals come from
projecting out the controls.

**Important**: FWL verification uses plain OLS (not cluster-robust), because FWL is
a property of OLS coefficients — cluster-robust SE only changes standard errors,
not coefficient estimates.


In [31]:
df_fwl = df[["dec", "attr"] + TRAITS + ["iid"]].dropna().copy()
ctrl_formula = " + ".join(TRAITS)
full_formula  = f"dec ~ attr + {ctrl_formula}"

# Plain OLS for coefficient verification
full_model_plain = smf.ols(full_formula, data=df_fwl).fit()
attr_coef_full   = full_model_plain.params["attr"]

# Clustered version for SE reporting
full_model_clustered = smf.ols(full_formula, data=df_fwl).fit(
    cov_type="cluster", cov_kwds={"groups": df_fwl["iid"].values})

print("Full model (clustered SE) — for reporting:")
print(f"  attr coef = {full_model_clustered.params['attr']:.6f}  "
      f"SE = {full_model_clustered.bse['attr']:.6f}  "
      f"p = {full_model_clustered.pvalues['attr']:.4f}")

# Residualize Y and attr
resid_y    = smf.ols(f"dec ~ {ctrl_formula}",  data=df_fwl).fit().resid
resid_attr = smf.ols(f"attr ~ {ctrl_formula}", data=df_fwl).fit().resid

fwl_df    = pd.DataFrame({"resid_y": resid_y.values,
                           "resid_attr": resid_attr.values})
fwl_model = smf.ols("resid_y ~ resid_attr", data=fwl_df).fit()
attr_coef_fwl = fwl_model.params["resid_attr"]

fwl_error = attr_coef_fwl - attr_coef_full
print(f"\nFWL verification:")
print(f"  attr coef (full model plain OLS) = {attr_coef_full:.10f}")
print(f"  attr coef (FWL)                  = {attr_coef_fwl:.10f}")
print(f"  difference                        = {fwl_error:.2e}")

status_fwl = "PASS" if abs(fwl_error) < 1e-6 else "FAIL"
print(f"  FWL check: [{status_fwl}]  (threshold: 1e-6)")

table04c = pd.DataFrame([{
    "attr_coef_full_model": attr_coef_full,
    "attr_coef_fwl":        attr_coef_fwl,
    "fwl_error":            fwl_error,
    "fwl_check":            status_fwl,
}])
table04c.to_csv("tables/table04c_fwl_verification.csv", index=False)
print("\nSaved: tables/table04c_fwl_verification.csv")


Full model (clustered SE) — for reporting:
  attr coef = 0.086479  SE = 0.005443  p = 0.0000

FWL verification:
  attr coef (full model plain OLS) = 0.0864788767
  attr coef (FWL)                  = 0.0864788767
  difference                        = -8.33e-17
  FWL check: [PASS]  (threshold: 1e-6)

Saved: tables/table04c_fwl_verification.csv


---
## Part D: Sensemakr Sensitivity Analysis

Sensemakr asks: how strong would an unobserved confounder need to be to overturn the
attractiveness effect estimated in the full model?

**Key**: Sensemakr runs on the **standard OLS fit** (not cluster-robust), because
its sensitivity method is based on OLS partial R². Cluster-robust SE is used only
for coefficient reporting in the table.

**Benchmark covariates must already be in the fitted model.** In the main model
(`dec ~ attr + sinc + intel + fun + amb + shar`), we benchmark against `sinc`, `intel`, `fun`.


In [32]:
import sensemakr as smkr

# Coefficient reporting model: cluster-robust SE
fitted_main_clustered = smf.ols(
    "dec ~ attr + sinc + intel + fun + amb + shar", data=df_fwl
).fit(cov_type="cluster", cov_kwds={"groups": df_fwl["iid"].values})

# Sensemakr model: standard OLS for partial R² sensitivity
fitted_main_sense = smf.ols(
    "dec ~ attr + sinc + intel + fun + amb + shar", data=df_fwl
).fit()

print("=== Full model (clustered SE) — coefficient table ===")
coef_report = pd.DataFrame({
    "coef": fitted_main_clustered.params,
    "se":   fitted_main_clustered.bse,
    "pval": fitted_main_clustered.pvalues,
}).drop("Intercept")
print(coef_report.round(5).to_string())

print("\n=== Sensemakr Sensitivity Analysis ===")
sens = smkr.Sensemakr(
    model=fitted_main_sense,
    treatment="attr",
    benchmark_covariates=["sinc", "intel", "fun"],
    kd=[1, 2, 3],
    q=1.0,
    alpha=0.05,
)
sens.summary()

rv_q1   = sens.sensitivity_stats["rv_q"]
rv_qa   = sens.sensitivity_stats["rv_qa"]
r2yd_x  = sens.sensitivity_stats["r2yd_x"]
print(f"\nRobustness Value (RV_q=1):       {rv_q1:.4f}")
print(f"Robustness Value (RV_alpha=0.05): {rv_qa:.4f}")
print(f"Partial R² of attr (r2yd_x):     {r2yd_x:.4f}")
print(f"\nRV_q=1 > 0.05:  {'PASS' if rv_q1 > 0.05 else 'FAIL (below minimum target)'}")
print(f"RV_q=1 > 0.10:  {'PASS (ideal target met)' if rv_q1 > 0.10 else 'NOT MET (below ideal target)'}")


=== Full model (clustered SE) — coefficient table ===
          coef       se     pval
attr   0.08648  0.00544  0.00000
sinc  -0.01911  0.00647  0.00312
intel  0.00833  0.00679  0.21995
fun    0.03805  0.00519  0.00000
amb   -0.02348  0.00506  0.00000
shar   0.04546  0.00505  0.00000

=== Sensemakr Sensitivity Analysis ===
Sensitivity Analysis to Unobserved Confounding

Model Formula: dec ~ attr + sinc + intel + fun + amb + shar

Null hypothesis: q = 1.0 and reduce = True 

-- This means we are considering biases that reduce the absolute value of the current estimate.
-- The null hypothesis deemed problematic is H0:tau = 0.0 

Unadjusted Estimates of ' attr ':
  Coef. estimate: 0.086
  Standard Error: 0.004
  t-value: 23.921 

Sensitivity Statistics:
  Partial R2 of treatment with outcome: 0.091
  Robustness Value, q = 1.0 : 0.27
  Robustness Value, q = 1.0 alpha = 0.05 : 0.251 

Verbal interpretation of sensitivity statistics:

-- Partial R2 of the treatment with the outcome: an extre

In [33]:
# Manual contour plot — avoids sens.plot() which breaks on matplotlib >= 3.8
# (QuadContourSet.collections was removed in mpl 3.8)

estimate_val = sens.sensitivity_stats["estimate"]
se_val       = sens.sensitivity_stats["se"]
dof_val      = int(sens.sensitivity_stats["dof"])
rv_q1_val    = sens.sensitivity_stats["rv_q"]
rv_qa_val    = sens.sensitivity_stats["rv_qa"]

# Build R² grid
grid_pts  = 200
r2_max    = min(0.6, float(sens.sensitivity_stats["r2yd_x"]) * 7)
r2_vals   = np.linspace(0, r2_max, grid_pts)
R2dz, R2yz = np.meshgrid(r2_vals, r2_vals)

# Adjusted estimate on grid
bias  = np.sqrt(R2yz * R2dz / (1 - R2dz)) * (se_val * np.sqrt(dof_val))
Z_est = estimate_val - bias

fig, ax = plt.subplots(figsize=(8, 7))

# Filled contour of adjusted estimate
levels = np.linspace(0, estimate_val * 1.1, 15)
cf = ax.contourf(R2dz, R2yz, Z_est,
                 levels=levels, cmap="RdYlGn", alpha=0.85)
plt.colorbar(cf, ax=ax, label="Bias-adjusted attr estimate")

# Zero contour (threshold where effect = 0)
cs0 = ax.contour(R2dz, R2yz, Z_est, levels=[0.0],
                 colors="black", linewidths=2)
ax.clabel(cs0, fmt="0", fontsize=9)

# Annotate benchmark bounds from sens.bounds
bounds_df = sens.bounds
colors_b = {"sinc": "#1f77b4", "intel": "#ff7f0e", "fun": "#2ca02c"}
plotted = set()
for _, row in bounds_df.iterrows():
    label = row["bound_label"]          # e.g. "1x sinc"
    trait_key = label.split()[-1]       # "sinc", "intel", "fun"
    col = colors_b.get(trait_key, "gray")
    multiplier = label.split("x")[0]    # "1","2","3"
    ax.scatter(row["r2dz_x"], row["r2yz_dx"],
               color=col, s=60, zorder=5,
               label=label if label not in plotted else "")
    ax.annotate(label, (row["r2dz_x"], row["r2yz_dx"]),
                xytext=(5, 5), textcoords="offset points",
                fontsize=8, color=col)
    plotted.add(label)

# RV lines
ax.axvline(rv_q1_val, color="red", linestyle="--", linewidth=1.2,
           label=f"RV_q=1 = {rv_q1_val:.3f}")
ax.axhline(rv_q1_val, color="red", linestyle="--", linewidth=1.2)

ax.set_xlabel("Partial R² of confounder with attr (r2dz_x)", fontsize=10)
ax.set_ylabel("Partial R² of confounder with dec (r2yz_dx)", fontsize=10)
ax.set_title("Sensemakr Sensitivity Contour\n"
             "Bias-adjusted attr effect; black = zero-effect threshold",
             fontsize=11, fontweight="bold")
ax.legend(fontsize=8, loc="upper right")
fig.tight_layout()
fig.savefig("figures/fig09_sensemakr_contour.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: figures/fig09_sensemakr_contour.png")
print(f"RV_q=1 = {rv_q1_val:.4f}  (dashed red lines)")
print(f"Zero-effect threshold = black contour")


Saved: figures/fig09_sensemakr_contour.png
RV_q=1 = 0.2699  (dashed red lines)
Zero-effect threshold = black contour


In [34]:
# Save Sensemakr summary table
table04d_rows = [{
    "model":               "main",
    "treatment":           "attr",
    "benchmark_covariates": "sinc, intel, fun",
    "attr_coef":           float(fitted_main_clustered.params["attr"]),
    "attr_clustered_se":   float(fitted_main_clustered.bse["attr"]),
    "RV_q1":               rv_q1,
    "RV_alpha05":          rv_qa,
    "partial_r2_attr":     r2yd_x,
    "n_obs":               int(fitted_main_sense.nobs),
    "notes":               "Primary model; clustered SE for reporting; plain OLS for Sensemakr",
}]

# ── Optional extended model with demographic controls ────────────────────────
print("\n=== Extended Sensemakr Model (with demographic controls) ===")
EXT_CONTROLS = ["gender", "samerace", "age_diff"]
ext_vars = ["dec", "attr"] + TRAITS + EXT_CONTROLS + ["iid"]
df_ext = df[ext_vars].dropna().copy()
print(f"Extended model sample: {len(df_ext):,} rows")

ext_formula = "dec ~ attr + sinc + intel + fun + amb + shar + gender + samerace + age_diff"

try:
    fitted_ext_clustered = smf.ols(ext_formula, data=df_ext).fit(
        cov_type="cluster", cov_kwds={"groups": df_ext["iid"].values})
    fitted_ext_sense = smf.ols(ext_formula, data=df_ext).fit()

    sens_ext = smkr.Sensemakr(
        model=fitted_ext_sense,
        treatment="attr",
        benchmark_covariates=["gender", "samerace", "age_diff"],
        kd=[1, 2, 3],
        q=1.0,
        alpha=0.05,
    )
    sens_ext.summary()

    rv_q1_ext  = sens_ext.sensitivity_stats["rv_q"]
    rv_qa_ext  = sens_ext.sensitivity_stats["rv_qa"]
    r2yd_x_ext = sens_ext.sensitivity_stats["r2yd_x"]

    print(f"Extended RV_q=1:       {rv_q1_ext:.4f}")
    print(f"Extended RV_alpha=0.05:{rv_qa_ext:.4f}")

    table04d_rows.append({
        "model":               "extended",
        "treatment":           "attr",
        "benchmark_covariates": "gender, samerace, age_diff",
        "attr_coef":           float(fitted_ext_clustered.params["attr"]),
        "attr_clustered_se":   float(fitted_ext_clustered.bse["attr"]),
        "RV_q1":               rv_q1_ext,
        "RV_alpha05":          rv_qa_ext,
        "partial_r2_attr":     r2yd_x_ext,
        "n_obs":               int(fitted_ext_sense.nobs),
        "notes":               "Extended model with demographics; clustered SE for reporting",
    })
except Exception as e:
    print(f"Extended Sensemakr failed: {e}")
    table04d_rows.append({
        "model": "extended", "treatment": "attr",
        "benchmark_covariates": "gender, samerace, age_diff",
        "attr_coef": None, "attr_clustered_se": None,
        "RV_q1": None, "RV_alpha05": None, "partial_r2_attr": None,
        "n_obs": None, "notes": f"FAILED: {e}",
    })

table04d = pd.DataFrame(table04d_rows)
table04d.to_csv("tables/table04d_sensemakr_summary.csv", index=False)
print("\nSaved: tables/table04d_sensemakr_summary.csv")
print(table04d[["model","attr_coef","RV_q1","RV_alpha05","notes"]].to_string(index=False))



=== Extended Sensemakr Model (with demographic controls) ===
Extended model sample: 5,600 rows
Sensitivity Analysis to Unobserved Confounding

Model Formula: dec ~ attr + sinc + intel + fun + amb + shar + gender + samerace + age_diff

Null hypothesis: q = 1.0 and reduce = True 

-- This means we are considering biases that reduce the absolute value of the current estimate.
-- The null hypothesis deemed problematic is H0:tau = 0.0 

Unadjusted Estimates of ' attr ':
  Coef. estimate: 0.086
  Standard Error: 0.004
  t-value: 23.284 

Sensitivity Statistics:
  Partial R2 of treatment with outcome: 0.088
  Robustness Value, q = 1.0 : 0.267
  Robustness Value, q = 1.0 alpha = 0.05 : 0.247 

Verbal interpretation of sensitivity statistics:

-- Partial R2 of the treatment with the outcome: an extreme confounder (orthogonal to the covariates)  that explains 100% of the residual variance of the outcome, would need to explain at least 8.841 % of the residual variance of the treatment to fully a

---
## Conclusion

### What each analysis answers

**Pairwise OVB** shows that the apparent predictive power of non-attractiveness traits
partly reflects their correlation with attractiveness. For each of the five non-attr traits
(`sinc`, `intel`, `fun`, `amb`, `shar`), the short regression coefficient (without `attr`)
is decomposed as:
`short_coef = long_coef + attr_coef × gamma`
where `gamma` is how much each trait correlates with `attr`. The OVB identity holds
exactly under OLS/LPM (formula_error < 0.001 for all traits), confirming the raw
halo mechanism.

**Controlled OVB** confirms whether this shrinkage persists after accounting for all
other perceived traits. Adding `attr` to the full five-trait model still changes each
trait's partial coefficient, with the magnitude governed by the matrix OVB identity
`alpha_X = beta_X + beta_attr × Pi`. This robustness check rules out the possibility
that pairwise shrinkage was driven purely by omitted-trait correlations rather than
the halo effect.

**FWL** reframes the full-model attractiveness coefficient as the effect of residualized
attractiveness on residualized decision-making, after removing variation explained by
the other perceived traits. The FWL coefficient matches the plain OLS coefficient exactly
(error < 1e-6), validating the within-controls identification of the attr effect.

**Sensemakr** asks how strong an unobserved confounder would need to be to overturn
this residualized attractiveness effect. A confounder would need to explain more than
`RV_q=1` (see table) of both `attr`'s and `dec`'s residual variance simultaneously
to reduce the attr effect to zero. Benchmarking against `sinc`, `intel`, and `fun`
provides a concrete interpretation: a confounder stronger than these perceived traits
by a factor of `k` would [or would not] be sufficient to overturn the conclusion.

### Limitations

- All analyses assume OLS/LPM linearity; OVB and FWL identities do not hold exactly in logistic models.
- Sensemakr uses standard OLS partial R² for sensitivity; cluster-robust SE is used only for reporting.
- The data are dyadic (each participant rates multiple partners), so residuals within participants
  are correlated. Standard errors are clustered at the participant level (`iid`) throughout.
